In [1]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (has pyproject.toml + quantdata/)."""
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists() and (d / "quantdata").is_dir():
            return d
    raise RuntimeError(f"quant_data project root not found above {start}")


PROJECT_ROOT = _find_root(Path.cwd().resolve())

# `quantdata` is an editable-installed package, so `import quantdata...` works
# already; PROJECT_ROOT is added so it also imports without the editable install.
# The `personal/` modules are flat scripts (no __init__.py), so every directory
# we import a module from must be on sys.path explicitly.
for p in (
    PROJECT_ROOT,
    PROJECT_ROOT / "personal" / "shared_functions",
    PROJECT_ROOT / "personal" / "SANOS_KF_Strike",
    PROJECT_ROOT / "personal" / "Kalman_filter_spline",
):
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import personal.SANOS_KF_Strike.sanos_ekf_eval as ekf

# personal/ flat modules:
import personal.shared_functions.unified_metrics as um

# quantdata package (example — import whatever you need):
import quantdata  # e.g. from quantdata.sanos import open as sanos_open
from personal.Kalman_filter_spline.surface_kalman_filters import build_iv_observations
from personal.SANOS_KF_Strike.sanos_kf_strike import (
    SanosKFStrikeConfig,
    gate_eval_table,
    run_sanos_kf_strike,
)
from quantdata.massive.client import MassiveClient

rows = []  # one metric dict per model, fed to um.comparison_table at the end

/Users/elie/Documents/GitHub/quant_data/quantdata/__init__.py:212: UserWarning: Using cache mode 'on' as specified in environment variable QUANTDATA_CACHE_MODE.


## CONFIG

In [2]:
TRADE_DATE = date(2024, 1, 3)
client = MassiveClient()
CALLS_ONLY, PUTS_ONLY = False, True
EVAL_IS_CALL = CALLS_ONLY if (CALLS_ONLY or PUTS_ONLY) else None

In [3]:
df_tickers = pl.read_csv(f"spy_options_tickers_{TRADE_DATE}.csv")

df_spy = client.fetch_trades("SPY", TRADE_DATE)

In [4]:
df_quotes, df_trades = build_iv_observations(df_tickers, TRADE_DATE, df_spy, min_trades=0)

Skipping O:SPY240105C00388000: no matched trades
Skipping O:SPY240105C00424000: no matched trades
Skipping O:SPY240105C00425000: no matched trades
Skipping O:SPY240105C00441000: no matched trades
Skipping O:SPY240105C00443000: no matched trades
Skipping O:SPY240105C00453000: no matched trades
Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105C00486000: 4860 bad quotes
Skipping O:SPY240105C00487000: 1630 bad quotes
Skipping O:SPY240105C00488000: 136 bad quotes
Skipping O:SPY240105C00489000: 130 bad quotes
Skipping O:SPY240105C00490000: 218 bad quotes
Skipping O:SPY240105C00491000: 370 bad quotes
Skipping O:SPY240105C00492000: 341 bad quotes
Skipping O:SPY240105C00493000: 342 bad quotes
Skipping O:SPY240105C00494000: 290 bad quotes
Skipping O:SPY240105C00495000: 202 bad quotes
Skipping O:SPY

In [5]:
import personal.Kalman_filter_spline.surface_model as sm

# df_quotes_iv / df_trades_iv are the IV-annotated outputs of build_iv_observations
# (already computed above as df_quotes / df_trades — just alias them here)
df_quotes_iv = df_quotes
df_trades_iv = df_trades
if EVAL_IS_CALL is not None:  # fit + eval on the same subset as the SANOS models (puts only)
    df_quotes_iv = df_quotes_iv.filter(pl.col("is_call") == EVAL_IS_CALL)
    df_trades_iv = df_trades_iv.filter(pl.col("is_call") == EVAL_IS_CALL)

In [6]:
grid_Q = [1e-8, 1e-6, 1e-4, 1e-2]  # process noise PER SECOND (batched fit ⇒ P_pred = P + Q·Δt)
grid_P0 = [1e-2, 1.0, 100]  # initial uncertainty (large → flat start, KF converges quickly)
Q_structures = ["diagonal"]  # , "scaled_identity"]  # per-param scaling via model.q_diag_weights

# Spline surface KF config
# SSVI_SLICE_MODEL: 3-param Gatheral SSVI, single-expiry (good for short-dated SPY)
# Swap for sm.QUAD_SLICE_MODEL (3 params) or sm.BSPLINE_SLICE_MODEL (8 params) to compare
cfg = sm.SurfaceModelConfig(
    model=sm.SSVI_SLICE_MODEL,
    Q_scale=1e-4,  # process noise PER SECOND (batched fit ⇒ P_pred = P + Q·Δt); retune vs per-tick
    Q_structure="diagonal",  # per-param scaling via model.q_diag_weights
    P0_scale=1e-3,  # initial uncertainty (large → flat start, KF converges quickly)
)

In [7]:
from personal.Kalman_filter_spline.surface_kalman_filters import run_spline_surface_kalman

for Q in grid_Q:
    for P0 in grid_P0:
        for Q_structure in Q_structures:
            cfg.Q_structure = Q_structure
            cfg.Q_scale = Q
            cfg.P0_scale = P0

            BATCH_S = (
                1.0  # fit & evaluate in 1-second batches (same methodology as the SANOS models)
            )

            q_res, t_res, df_state = run_spline_surface_kalman(
                df_quotes_iv, df_trades_iv, cfg, batch_seconds=BATCH_S
            )
            q = q_res.sort(["strike", "is_call", "ts_ns"]).to_pandas()

            # 1 s window id; within a window the spline shares one posterior θ and one held-out (previous-
            # window) prior θ, so prices vary only through each quote's own forward/T/k. Take the LAST quote
            # of each (strike, is_call) in the window as the per-second snapshot representative, then group by
            # the window — one cross-sectional snapshot per 1 s bucket, exactly like the SANOS 2-stage table.
            q["window"] = q["ts_ns"].astype("int64") // int(BATCH_S * 1e9)
            # snap = q.sort_values("ts_ns").drop_duplicates(subset=["window", "strike", "is_call"], keep="last")
            # held-out prior is the batched filter's own pre-update prediction; drop any row where it is
            # undefined (e.g. invalid forward/T at the snapshot quote).
            snap = q
            snap = snap.dropna(subset=["kalman_price_prior"])

            df_spline = um.from_option_prices(
                ts=snap[
                    "window"
                ],  # → n_snapshots = number of 1 s buckets (~24k), cross-sectional over strikes
                strike=snap["strike"],
                forward=snap["forward"],
                is_call=snap["is_call"],
                bid=snap["bid_price"],
                ask=snap["ask_price"],
                prior_price=snap["kalman_price_prior"],
                post_price=snap["kalman_price"],
                discount_factor=snap["discount_factor"] if "discount_factor" in snap else None,
            )
            # ── Negative mass: reconstruct smile over a strike grid at sampled snapshot times ──
            state_pd = df_state.to_pandas().sort_values("ts_ns").reset_index(drop=True)
            n_params = cfg.n_params
            theta_cols = [f"theta_{j}" for j in range(n_params)]

            # Strike grid spanning the observed strikes
            K_lo = float(df_quotes_iv["strike"].min())
            K_hi = float(df_quotes_iv["strike"].max())
            grid_K = np.linspace(K_lo, K_hi, 41)

            # Subsample snapshot times for speed (1 000 is usually plenty)
            n_snap = min(1000, len(state_pd))
            samp_idx = np.linspace(0, len(state_pd) - 1, n_snap, dtype=int)
            state_samp = state_pd.iloc[samp_idx].reset_index(drop=True)

            # For each sampled ts: look up the most-recent forward/T from the quote stream
            q_sorted = q.sort_values("ts_ns").reset_index(drop=True)
            asof = (
                np.searchsorted(q_sorted["ts_ns"].values, state_samp["ts_ns"].values, side="right")
                - 1
            )
            asof = np.clip(asof, 0, len(q_sorted) - 1)
            F_vec = q_sorted["forward"].values[asof]
            T_vec = q_sorted["T"].values[asof]
            thetas_samp = state_samp[theta_cols].values  # (n_snap, n_params)

            # Reconstruct call-price matrix (n_snap, n_grid)
            C_mat = np.full((n_snap, len(grid_K)), np.nan)
            for i in range(n_snap):
                theta_i, F_i, T_i = thetas_samp[i], F_vec[i], T_vec[i]
                if not (np.isfinite(theta_i).all() and F_i > 0 and np.isfinite(F_i) and T_i > 0):
                    continue
                ivs = np.array(
                    [cfg.model.observe(theta_i, np.log(k / F_i), T_i)[0] for k in grid_K]
                )
                C_mat[i] = um.bs_call_price(
                    np.full(len(grid_K), F_i), grid_K, np.full(len(grid_K), T_i), ivs
                )

            valid = np.isfinite(C_mat).all(axis=1)
            nm = um.negative_mass_surface(grid_K, C_mat[valid], F_vec[valid])

            m_spline = um.evaluate(
                df_spline,
                model_name=f"KF spline-{cfg.Q_scale}-{cfg.P0_scale}-{cfg.Q_structure}",
                neg_mass_values=nm,
            )
            rows.append(m_spline)
            print(m_spline)

{'model': 'KF spline-1e-08-0.01-diagonal', 'ho_nse_mean': 1.252633053942333, 'ho_nse_med': 1.0352826397762294, 'ho_nse_std': 2.918581603709253, 'is_nse_mean': 1.0323358160764642, 'is_nse_med': 0.9422694978562377, 'is_nse_std': 0.8889462635202223, 'ho_rmse_mean': 0.0802533465189839, 'ho_rmse_med': 0.07583199867305311, 'ho_rmse_std': 0.06040343698823898, 'is_rmse_mean': 0.07899368416920306, 'is_rmse_med': 0.07541291630750459, 'is_rmse_std': 0.05737885689980278, 'ho_out_pct_mean': 42.15975647595696, 'ho_out_pct_med': 41.23711340206185, 'is_out_pct_mean': 36.5775711572695, 'is_out_pct_med': 35.44303797468354, 'neg_mass_mean': 0.0, 'neg_mass_med': 0.0, 'arb_frac': 0.0, 'n_snapshots': 24119}
{'model': 'KF spline-1e-08-1.0-diagonal', 'ho_nse_mean': 1.3029528632883447, 'ho_nse_med': 1.1366285905891345, 'ho_nse_std': 2.7238251483054188, 'is_nse_mean': 1.0783976157824076, 'is_nse_med': 1.0396385838684403, 'is_nse_std': 0.7819460836344302, 'ho_rmse_mean': 0.07517153139791664, 'ho_rmse_med': 0.068

In [8]:
df = um.comparison_table(rows)

In [9]:
df.sort_values("ho_nse_med").to_csv(f"spy_options_kf_spline_eval_{TRADE_DATE}.csv", index=True)

In [10]:
df.sort_values("ho_nse_med")

,ho_nse_mean,ho_nse_med,ho_nse_std,is_nse_mean,is_nse_med,is_nse_std,ho_rmse_mean,ho_rmse_med,ho_rmse_std,is_rmse_mean,is_rmse_med,is_rmse_std,ho_out_pct_mean,ho_out_pct_med,is_out_pct_mean,is_out_pct_med,neg_mass_mean,neg_mass_med,arb_frac,n_snapshots
model,,,,,,,,,,,,,,,,,,,,
KF spline-1e-08-0.01-diagonal,1.252633,1.035283,2.918582,1.032336,0.942269,0.888946,0.080253,0.075832,0.060403,0.078994,0.075413,0.057379,42.159756,41.237113,36.577571,35.443038,0.000000e+00,0.0,0.0,24119
KF spline-1e-06-0.01-diagonal,1.314512,1.093757,3.709842,1.071475,0.993746,1.558409,0.077202,0.071155,0.061743,0.075742,0.070749,0.057153,45.720508,45.454545,40.079482,40.000000,0.000000e+00,0.0,0.0,24119
KF spline-0.0001-0.01-diagonal,1.334242,1.115419,3.781664,1.062166,0.990350,1.568892,0.075988,0.069360,0.061420,0.074380,0.068961,0.056617,47.322285,47.500000,40.704136,40.384615,0.000000e+00,0.0,0.0,24119
KF spline-0.01-1.0-diagonal,1.363608,1.122954,3.809811,1.013627,0.948134,1.565831,0.075526,0.068986,0.060847,0.073401,0.068353,0.056149,47.300122,47.058824,38.475417,37.906137,0.000000e+00,0.0,0.0,24119
KF spline-0.01-0.01-diagonal,1.365458,1.123730,3.810126,1.015456,0.947507,1.566772,0.075749,0.069103,0.060968,0.073624,0.068542,0.056287,47.333138,47.058824,38.510776,37.899543,0.000000e+00,0.0,0.0,24119
KF spline-0.0001-1.0-diagonal,1.338255,1.124524,3.778565,1.065293,0.998303,1.567157,0.075194,0.068509,0.061024,0.073586,0.068137,0.056182,47.764568,48.051948,41.221434,41.176471,0.000000e+00,0.0,0.0,24119
KF spline-0.01-100-diagonal,1.368718,1.125793,3.815974,1.018123,0.949340,1.577728,0.075545,0.068971,0.061030,0.073418,0.068313,0.056344,47.382616,47.244094,38.562357,38.000000,-1.663114e-16,0.0,0.0,24119
KF spline-1e-06-1.0-diagonal,1.335202,1.128972,3.679375,1.090749,1.030604,1.548285,0.075322,0.068740,0.060826,0.073856,0.068344,0.056219,47.935005,48.292683,42.523399,42.857143,0.000000e+00,0.0,0.0,24119
KF spline-1e-08-1.0-diagonal,1.302953,1.136629,2.723825,1.078398,1.039639,0.781946,0.075172,0.068855,0.058042,0.073879,0.068452,0.055268,47.997906,48.076923,42.603995,43.023256,0.000000e+00,0.0,0.0,24119


In [11]:
rows

[{'model': 'KF spline-1e-08-0.01-diagonal',
  'ho_nse_mean': 1.252633053942333,
  'ho_nse_med': 1.0352826397762294,
  'ho_nse_std': 2.918581603709253,
  'is_nse_mean': 1.0323358160764642,
  'is_nse_med': 0.9422694978562377,
  'is_nse_std': 0.8889462635202223,
  'ho_rmse_mean': 0.0802533465189839,
  'ho_rmse_med': 0.07583199867305311,
  'ho_rmse_std': 0.06040343698823898,
  'is_rmse_mean': 0.07899368416920306,
  'is_rmse_med': 0.07541291630750459,
  'is_rmse_std': 0.05737885689980278,
  'ho_out_pct_mean': 42.15975647595696,
  'ho_out_pct_med': 41.23711340206185,
  'is_out_pct_mean': 36.5775711572695,
  'is_out_pct_med': 35.44303797468354,
  'neg_mass_mean': 0.0,
  'neg_mass_med': 0.0,
  'arb_frac': 0.0,
  'n_snapshots': 24119},
 {'model': 'KF spline-1e-08-1.0-diagonal',
  'ho_nse_mean': 1.3029528632883447,
  'ho_nse_med': 1.1366285905891345,
  'ho_nse_std': 2.7238251483054188,
  'is_nse_mean': 1.0783976157824076,
  'is_nse_med': 1.0396385838684403,
  'is_nse_std': 0.7819460836344302,
  